In [35]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib import style
style.use('ggplot')
import mplfinance as mpf
import yfinance as yf
from datetime import datetime, timedelta

In [36]:
TICKERS     = ["AAPL", "MSFT", "GOOGL", "AMZN", "NVDA"]
COLORS      = ["#2196F3", "#4CAF50", "#FF5722", "#9C27B0", "#FF9800"]
END_DATE    = datetime.today()
START_DATE  = END_DATE - timedelta(days=365)

In [37]:
def fetch_data(tickers, start, end):
    """Return a dict {ticker: OHLCV DataFrame} using yfinance"""
    try:
        raw = yf.download(tickers, start=start, end=end,
                          auto_adjust=True, progress=False)
        if raw.empty:
            raise ValueError("Empty download")

        result = {}
        for t in tickers:
            df = raw.xs(t, axis=1, level=1)[["Open","High","Low","Close","Volume"]].dropna()
            result[t] = df
        print(f"✓ Live data fetched for {', '.join(tickers)}")
        return result

    except Exception as e:
        print(f"ℹ  yfinance unavailable ({e})")
        return

In [38]:
def compute_metrics(data: dict) -> pd.DataFrame:
    rows = []
    for ticker, df in data.items():
        daily_ret   = df["Close"].pct_change().dropna()
        ann_return  = (1 + daily_ret.mean()) ** 252 - 1
        ann_vol     = daily_ret.std() * np.sqrt(252)
        total_ret   = (df["Close"].iloc[-1] / df["Close"].iloc[0]) - 1
        sharpe      = ann_return / ann_vol if ann_vol else np.nan
        rows.append({
            "Ticker":           ticker,
            "Start Price":      f"${df['Close'].iloc[0]:,.2f}",
            "End Price":        f"${df['Close'].iloc[-1]:,.2f}",
            "Total Return":     f"{total_ret:+.1%}",
            "Ann. Return":      f"{ann_return:+.1%}",
            "Ann. Volatility":  f"{ann_vol:.1%}",
            "Sharpe (approx)":  f"{sharpe:.2f}",
        })
    return pd.DataFrame(rows).set_index("Ticker")

In [39]:
def plot_line(data: dict, save_path="chart_line.png"):
    fig, axes = plt.subplots(2, 1, figsize=(14, 9),
                             gridspec_kw={"height_ratios": [3, 1]})
    ax, ax_vol = axes

    # Normalised closing prices (base 100)
    ax_norm = ax.twinx()
    for (ticker, df), color in zip(data.items(), COLORS):
        norm = df["Close"] / df["Close"].iloc[0] * 100
        ax_norm.plot(df.index, norm, color=color, linewidth=1.8,
                     label=f"{ticker} (norm)", linestyle="--", alpha=0.45)
        ax.plot(df.index, df["Close"], color=color, linewidth=2,
                label=ticker)

    ax.set_title("1-Year Closing Prices – Absolute & Normalised (dashed)",
                 fontsize=15, fontweight="bold", pad=12)
    ax.set_ylabel("Price (USD)", fontsize=11)
    ax_norm.set_ylabel("Indexed (100 = start)", fontsize=10, color="grey")
    ax_norm.tick_params(axis="y", labelcolor="grey")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    ax.legend(loc="upper left", fontsize=10)
    ax.grid(alpha=0.25)

    # Volume bars (average across tickers)
    avg_vol = pd.concat([df["Volume"] for df in data.values()], axis=1).mean(axis=1)
    ax_vol.bar(avg_vol.index, avg_vol / 1e6, color="#90CAF9", width=1, alpha=0.7)
    ax_vol.set_ylabel("Avg Vol (M)", fontsize=9)
    ax_vol.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
    ax_vol.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    ax_vol.grid(alpha=0.2)

    plt.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved → {save_path}")


# ── Chart 2 – Candlestick (mplfinance multi-panel) ────────────────────────────

def plot_candlestick(data: dict, save_path="chart_candlestick.png"):
    """Candlestick for each ticker, last ~90 trading days for clarity."""
    tickers = list(data.keys())
    n = len(tickers)
    fig, axes = plt.subplots(n, 1, figsize=(14, 4 * n), sharex=False)

    style = mpf.make_mpf_style(
        base_mpf_style="nightclouds",
        marketcolors=mpf.make_marketcolors(up="#26a69a", down="#ef5350",
                                           edge="inherit", wick="inherit",
                                           volume="in"),
        facecolor="#1e1e2e", figcolor="#1e1e2e",
        gridstyle=":", gridcolor="#444"
    )

    for ax, ticker in zip(axes, tickers):
        df = data[ticker].tail(90).copy()
        df.index = pd.DatetimeIndex(df.index)

        mpf.plot(df, type="candle", ax=ax, style=style,
                 show_nontrading=False, warn_too_much_data=10000)
        ax.set_title(f"{ticker} – Last 90 Days Candlestick",
                     fontsize=12, fontweight="bold", color="#e0e0e0")
        ax.set_ylabel("Price (USD)", color="#ccc")
        ax.tick_params(colors="#ccc")
        ax.set_facecolor("#1e1e2e")

    fig.patch.set_facecolor("#1e1e2e")
    fig.suptitle("Candlestick Charts – Last 90 Trading Days",
                 fontsize=15, fontweight="bold", color="#e0e0e0", y=1.005)
    plt.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.close(fig)
    print(f"  Saved → {save_path}")


In [40]:
def main():
    print("=" * 55)
    print("  Quant ML Journey – Stock Analysis")
    print("=" * 55)

    data = fetch_data(TICKERS, START_DATE, END_DATE)

    print("\n── Metrics ──────────────────────────────────────────")
    metrics = compute_metrics(data)
    print(metrics.to_string())
    metrics.to_csv("metrics.csv")
    print("\n  Saved → metrics.csv")

    print("\n── Generating charts ────────────────────────────────")
    plot_line(data)
    plot_candlestick(data)

    print("\n✓ All done!  Check the PNG files in this directory.")
    print("=" * 55)